# Generalization validation: remaining 810 SQuAD questions

This follow-up was planned **after seeing the earlier 90-question results**. The earlier test found a LoRA-minus-Baseline EM change of **+5.56 percentage points**, with a 95% article-cluster bootstrap interval of **−3.89 to +15.56 pp**. The original predeclared prediction that LoRA EM would fall by at least 8 pp was not met on that 90-question point estimate. Those results are retained unchanged and must be reported separately.

**Frozen validation plan, before running these 810 questions:** Use the exact complement of the earlier two-per-article sample within the published 900-question SQuAD 1.1 test set: 18 further questions from each of the same 45 articles. Use the same published Qwen3-0.6B base model, seed-17 epoch-2 LoRA adapter, prompt, greedy generation with a 64-token cap, distractor construction, and fixed distractor seeds 101 and 202. Evaluate only original context and ~1,024 irrelevant tokens **prepended** to it. No training is performed. The primary estimand is the LoRA EM change minus the Baseline EM change, averaging the two distractor seeds for each question, then pairing the models on each question. Calculate a 95% percentile interval by resampling the 45 articles with replacement (2,000 draws, RNG seed 42). A positive point estimate with lower interval bound above zero supports the narrow statement that this checkpoint's EM fell less than Baseline's in this SQuAD prepended-context setting. Report both seeds, individual model changes, F1, and all failures regardless of outcome. This is follow-up validation on new questions from the **same articles**, not cross-domain generalization or a new article test.

Run top to bottom in Google Colab with a GPU. Output goes to a separate Drive folder so the original 90-question run and later placement control are preserved.


In [ ]:
# 1. Setup. Colab's installed torch is retained; install the training notebook's library versions.
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # Must precede importing torch.
%pip -q install "transformers==4.53.3" "tokenizers==0.21.4" "peft==0.17.1" "accelerate==1.8.1" "huggingface_hub==0.36.2" "numpy==2.2.6" "safetensors==0.8.0" "pandas==2.2.3" "matplotlib==3.10.0"

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess
ROOT = Path('/content/LLM-project')
PINNED_COMMIT = '2359fd5d9ea34bb9d1845fe1a6732d657343361f'
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/sunweiyi2026-ship-it/LLM-project.git', str(ROOT)], check=True)
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', PINNED_COMMIT], check=True)
subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', PINNED_COMMIT], check=True)
REPO_COMMIT = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
assert REPO_COMMIT == PINNED_COMMIT
OUTPUT_DIR = Path('/content/drive/MyDrive/LLM-project/generalization_810_prepend_seed17_epoch2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert (ROOT / 'published/seed17_epoch2/manifest.json').is_file()
print('Published model and data:', ROOT / 'published/seed17_epoch2', '| commit:', REPO_COMMIT)
print('Outputs:', OUTPUT_DIR)


In [ ]:
# 2. Verify the published 900-question list and load the pinned model + adapter.
import hashlib, json, sys
from collections import defaultdict
from contextlib import nullcontext
import numpy as np
import pandas as pd
import torch
sys.path.insert(0, str(ROOT / 'src'))
from genqa.pretrained import verify_published, load_published_model
from genqa import engine

assert torch.cuda.is_available(), 'Choose a GPU runtime in Colab.'
verification = verify_published(ROOT)
loaded = load_published_model(ROOT)
publication = loaded['spec']
test_rows = loaded['rows']['test']
original_ids = json.loads((ROOT / 'published/seed17_epoch2/reference/sample_ids.json').read_text())['test']
assert len(test_rows) == len(original_ids) == 900
assert [r['id'] for r in test_rows] == original_ids, 'Test IDs/order differ from original run'
assert len({r['article_title'] for r in test_rows}) == 45
model, tokenizer = loaded['model'], loaded['tokenizer']
MODEL_ID = publication['model']['model_id']
REVISION = publication['model']['revision']
PROMPT = loaded['prompt']['instruction']
MAX_NEW_TOKENS = publication['evaluation']['max_new_tokens']
BATCH_SIZE = publication['evaluation']['batch_size']
SAMPLE_SEED = 731
DISTRACTOR_SEEDS = (101, 202)
TARGET_EXTRA_TOKENS = (1024,)
print('Verified:', verification['status'], '| 900 IDs:', len(original_ids),
      '| GPU:', torch.cuda.get_device_name(0), '| source run:', publication['source_run']['name'])
reference_metrics = pd.read_csv(ROOT / 'published/seed17_epoch2/reference/metrics.csv')
display(reference_metrics[reference_metrics['phase'].eq('test')][['method', 'phase', 'n', 'EM (%)', 'F1 (%)']])


In [ ]:
# 3. Freeze the exact 810-question complement before any new inference.
by_article = defaultdict(list)
paragraphs = defaultdict(list)
for row in test_rows:
    title = row['article_title']
    paragraphs[title].append(row['context'])
    by_article[title].append({**row, 'article': title})

def stable_key(value):
    return hashlib.sha256(f'{SAMPLE_SEED}:{value}'.encode()).hexdigest()

prior_90 = []
selected = []
for title in sorted(by_article):
    ordered = sorted(by_article[title], key=lambda r: stable_key(r['id']))
    assert len(ordered) == 20, f'Unexpected question count in {title}'
    prior_90.extend(ordered[:2])
    selected.extend(ordered[2:])
prior_ids = {r['id'] for r in prior_90}
assert len(prior_ids) == 90 and len(selected) == 810
assert not prior_ids.intersection(r['id'] for r in selected)
assert prior_ids.union(r['id'] for r in selected) == set(original_ids)
assert len({r['id'] for r in selected}) == len(selected)
print('New questions:', len(selected), '| previous questions:', len(prior_90), '| articles:', len(by_article))

pool = [(title, context) for title in sorted(paragraphs)
        for context in sorted(set(paragraphs[title]))]

def context_tokens(text):
    return tokenizer.encode(text, add_special_tokens=False)

def distractor(row, target, seed):
    # Identical construction to the prior 90-question notebook.
    key = int(hashlib.sha256(f'{seed}:{row["id"]}'.encode()).hexdigest()[:16], 16)
    rng = np.random.default_rng(key)
    order = rng.permutation(len(pool))
    tokens = []
    answer_texts = [a.lower().strip() for a in row['answers'] if len(a.strip()) >= 4]
    for index in order:
        title, text = pool[int(index)]
        if title == row['article'] or text == row['context']:
            continue
        if any(a in text.lower() for a in answer_texts):
            continue
        tokens.extend(context_tokens(text + '\n\n'))
        if len(tokens) >= target:
            return tokenizer.decode(tokens[:target], skip_special_tokens=True).strip()
    raise RuntimeError('Insufficient distractor text')

conditions = [('original', 0, None)] + [
    (f'extra_1024_seed_{seed}', 1024, seed) for seed in DISTRACTOR_SEEDS
]
cases = []
for row in selected:
    for condition, extra, seed in conditions:
        added = distractor(row, extra, seed) if extra else ''
        context = f'{added}\n\n{row["context"]}' if added else row['context']
        cases.append({**row, 'condition': condition, 'extra_target_tokens': extra,
                      'distractor_seed': seed, 'eval_context': context})
assert len(cases) == 810 * 3
print('Input cases per model:', len(cases), '| total generations:', len(cases) * 2)

for row in cases:
    prepared = engine.prepare_input({**row, 'context': row['eval_context']}, tokenizer,
                                    loaded['prompt'], publication['evaluation']['max_input_tokens'])
    assert not prepared['context_truncated'], f'Input was truncated: {row["id"]}'
    row['prompt_tokens'] = prepared['input_tokens']

manifest = {
    'kind': 'follow_up_810_prepend_validation',
    'design_status': 'frozen before inference on these 810 questions; motivated by prior 90-question results',
    'prior_90_result': {'LoRA_minus_Baseline_EM_change_pp': 5.555555555555555,
                        'ci95_pp': [-3.888888888888889, 15.555555555555555],
                        'original_prediction_LoRA_drop_at_least_8pp_met': False},
    'primary_estimand': 'LoRA minus Baseline EM change from original to +1024 prepended distractor, paired by question, two seeds averaged per question',
    'decision_rule': 'Narrow relative robustness supported if primary estimate > 0 and 95% article-cluster bootstrap lower bound > 0',
    'bootstrap': {'cluster': 'article', 'draws': 2000, 'seed': 42, 'interval': 'percentile'},
    'publication': 'published/seed17_epoch2/manifest.json',
    'repository_commit': REPO_COMMIT, 'publication_verification': verification,
    'test_rows_sha256': publication['data']['test']['rows_sha256'],
    'model_id': MODEL_ID, 'revision': REVISION,
    'adapter_sha256': publication['files']['adapter/adapter_model.safetensors']['sha256'],
    'source_run': publication['source_run'], 'sample_seed': SAMPLE_SEED,
    'prior_question_ids': [r['id'] for r in prior_90],
    'question_ids': [r['id'] for r in selected],
    'distractor_seeds': DISTRACTOR_SEEDS, 'target_extra_tokens': TARGET_EXTRA_TOKENS,
    'article_count': len(by_article), 'questions_per_article': 18,
    'prompt': PROMPT,
    'generation': {'do_sample': False, 'max_new_tokens': MAX_NEW_TOKENS,
                   'enable_thinking': False, 'batch_size': BATCH_SIZE},
    'max_prompt_tokens': max(r['prompt_tokens'] for r in cases)
}
manifest_path = OUTPUT_DIR / 'manifest.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest, 'Output directory belongs to a different experiment'
else:
    manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
print('Plan saved:', manifest_path)


In [ ]:
# 4. Evaluate 810 questions with the original generation and scoring engine.
# This keeps prompt tokenization, EOS handling, greedy generation, and SQuAD scoring
# aligned with the published run. Completed conditions are reused after a disconnect.
import shutil, time

def result_path(method, condition):
    return OUTPUT_DIR / method.lower() / condition / 'predictions.jsonl'

for method in ('Baseline', 'LoRA'):
    for condition, extra, seed in conditions:
        rows = [{**r, 'context': r['eval_context']} for r in cases if r['condition'] == condition]
        destination = result_path(method, condition).parent
        if destination.exists():
            metrics_path = destination / 'metrics.json'
            metrics = json.loads(metrics_path.read_text()) if metrics_path.is_file() else {}
            if not metrics.get('complete'):
                archived = destination.with_name(destination.name + f'_interrupted_{int(time.time())}')
                shutil.move(str(destination), str(archived))
                print('Archived incomplete condition:', archived)
            else:
                assert metrics['n_predictions'] == len(rows)
                completed = [json.loads(line) for line in result_path(method, condition).read_text().splitlines() if line]
                assert {r['id'] for r in completed} == {r['id'] for r in rows}
                print('Reused complete condition:', method, condition)
                continue
        print('Evaluating:', method, condition, len(rows), 'questions', flush=True)
        adapter_context = model.disable_adapter() if method == 'Baseline' else nullcontext()
        with adapter_context:
            engine.evaluate(model, tokenizer, rows, loaded['prompt'], destination,
                            batch_size=BATCH_SIZE,
                            max_input_tokens=publication['evaluation']['max_input_tokens'],
                            max_new_tokens=MAX_NEW_TOKENS, loss_examples_per_split=0,
                            seed=publication['evaluation']['seed'],
                            run_label=f'Generalization {method} {condition}')


In [ ]:
# 5. Analyze completed predictions. This cell also works after a runtime restart.
# It reads the frozen manifest and six completed prediction files; no inference runs here.
import json, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd

OUTPUT_DIR = Path(globals().get('OUTPUT_DIR', '/content/drive/MyDrive/LLM-project/generalization_810_prepend_seed17_epoch2'))
if not Path('/content/drive/MyDrive').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
manifest_path = OUTPUT_DIR / 'manifest.json'
assert manifest_path.is_file(), f'Missing manifest: {manifest_path}. Run cells 1-3 first.'
manifest = json.loads(manifest_path.read_text())
assert manifest['kind'] == 'follow_up_810_prepend_validation'
selected = manifest['question_ids']
assert len(selected) == 810 and len(set(selected)) == 810
seeds = tuple(manifest['distractor_seeds'])
assert seeds == (101, 202) and tuple(manifest['target_extra_tokens']) == (1024,)
conditions = [('original', 0, None)] + [(f'extra_1024_seed_{seed}', 1024, seed) for seed in seeds]

def result_path(method, condition):
    return OUTPUT_DIR / method.lower() / condition / 'predictions.jsonl'

required = [(method, condition) for method in ('Baseline', 'LoRA')
            for condition, _, _ in conditions]
incomplete = []
for method, condition in required:
    path = result_path(method, condition)
    metrics_path = path.parent / 'metrics.json'
    if not path.is_file() or not metrics_path.is_file():
        incomplete.append(f'{method}/{condition}')
        continue
    metrics = json.loads(metrics_path.read_text())
    if not metrics.get('complete') or metrics.get('n_predictions') != 810:
        incomplete.append(f'{method}/{condition}')
assert not incomplete, 'Inference is not complete for: ' + ', '.join(incomplete) + '. Run cell 4 first.'

ROOT = Path('/content/LLM-project')
PINNED_COMMIT = manifest['repository_commit']
if not (ROOT / 'src/genqa/engine.py').is_file():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/sunweiyi2026-ship-it/LLM-project.git', str(ROOT)], check=True)
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', PINNED_COMMIT], check=True)
subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', PINNED_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip() == PINNED_COMMIT
sys.path.insert(0, str(ROOT / 'src'))
from genqa import engine

records = []
for method in ('Baseline', 'LoRA'):
    for condition, _, _ in conditions:
        with result_path(method, condition).open() as stream:
            for line in stream:
                if line.strip():
                    row = json.loads(line)
                    score = engine.score_answer(row['prediction'], row['answers'])
                    records.append({**row, 'method': method,
                                    'em': score['em'] / 100, 'f1': score['f1'] / 100,
                                    'prompt_tokens': row['input_tokens']})
pred = pd.DataFrame(records)
assert len(pred) == 2 * len(conditions) * len(selected) == 4860
assert not pred.duplicated(['method', 'condition', 'id']).any()
assert set(pred.id) == set(manifest['question_ids'])
assert not pred.context_truncated.any()
pred.to_csv(OUTPUT_DIR / 'all_predictions.csv', index=False)

summary = (pred.groupby(['method', 'extra_target_tokens', 'distractor_seed'], dropna=False)
           .agg(n=('id', 'count'), articles=('article', 'nunique'),
                em=('em', 'mean'), f1=('f1', 'mean'),
                mean_prompt_tokens=('prompt_tokens', 'mean')).reset_index())
summary[['em', 'f1']] *= 100
summary.to_csv(OUTPUT_DIR / 'condition_summary.csv', index=False)
display(summary.round(2))

def question_changes(method, metric):
    subset = pred[pred.method.eq(method)]
    before = subset[subset.extra_target_tokens.eq(0)].set_index('id')[metric]
    after = (subset[subset.extra_target_tokens.eq(1024)]
             .groupby(['article', 'id'])[metric].mean().reset_index())
    after['change'] = after[metric] - after['id'].map(before)
    assert after.change.notna().all() and len(after) == len(selected)
    return after.set_index(['article', 'id'])['change'].sort_index()

def article_interval(question_values):
    article_means = question_values.groupby(level='article').mean().to_numpy()
    assert len(article_means) == 45
    rng = np.random.default_rng(42)
    draws = rng.choice(article_means, size=(2000, len(article_means)), replace=True).mean(axis=1)
    low, high = np.quantile(draws, [.025, .975])
    return 100 * article_means.mean(), 100 * low, 100 * high

rows = []
for metric in ('em', 'f1'):
    baseline = question_changes('Baseline', metric)
    lora = question_changes('LoRA', metric)
    assert baseline.index.equals(lora.index)
    for contrast, values in [('Baseline', baseline), ('LoRA', lora),
                             ('LoRA_minus_Baseline', lora - baseline)]:
        estimate, low, high = article_interval(values)
        rows.append({'metric': metric, 'contrast': contrast, 'change_pp': estimate,
                     'ci95_low_pp': low, 'ci95_high_pp': high,
                     'questions': len(values), 'articles': 45, 'bootstrap_draws': 2000})
effects = pd.DataFrame(rows)
effects.to_csv(OUTPUT_DIR / 'paired_effects.csv', index=False)
display(effects.round(2))

base_lora = pred[(pred.method.eq('LoRA')) & (pred.extra_target_tokens.eq(0))]
stressed_lora = pred[(pred.method.eq('LoRA')) & (pred.extra_target_tokens.eq(1024))]
failures = stressed_lora.merge(base_lora[['id', 'em', 'prediction']].rename(
    columns={'em': 'base_em', 'prediction': 'base_prediction'}),
    on='id', validate='many_to_one')
failures = failures[(failures.base_em.eq(1)) & (failures.em.eq(0))].sort_values(
    ['distractor_seed', 'article', 'id'])
failures[['id', 'article', 'question', 'answers', 'distractor_seed',
          'base_prediction', 'prediction']].to_csv(OUTPUT_DIR / 'lora_failures_1024.csv', index=False)

primary = effects[(effects.metric.eq('em')) &
                  (effects.contrast.eq('LoRA_minus_Baseline'))].iloc[0]
supports_narrow_claim = bool(primary.change_pp > 0 and primary.ci95_low_pp > 0)
report = f'''# 810-question follow-up: prepended irrelevant context

This follow-up was designed after the earlier 90-question result. The earlier comparison was +5.56 pp, with a 95% article-cluster bootstrap interval from -3.89 to +15.56 pp. Its original prediction of an 8 pp LoRA EM decline was not met by the point estimate. Those findings remain unchanged.

The 810 new questions are the exact complement within the published 900-question SQuAD 1.1 test set: 18 new questions in each of the same 45 articles. The same frozen base model, epoch-2 LoRA adapter, prompt, greedy decoding, and distractor seeds 101 and 202 were used. All model inputs were untruncated. No training was performed.

Primary LoRA-minus-Baseline EM change from original to +1024 prepended tokens: {primary.change_pp:+.2f} pp (95% article-cluster bootstrap interval {primary.ci95_low_pp:+.2f} to {primary.ci95_high_pp:+.2f} pp). Prespecified narrow criterion met: {supports_narrow_claim}.

This result concerns this checkpoint and SQuAD prepended-context setting. The 810 questions come from the same 45 articles as the first 90, so it is not evidence of transfer to new articles or domains. The two distractor seeds are fixed; the bootstrap resamples articles, not seeds. See condition_summary.csv, paired_effects.csv, all_predictions.csv, and lora_failures_1024.csv.
'''
(OUTPUT_DIR / 'report.md').write_text(report)
print(report)
